In [1]:
import os
import pandas as pd
from datetime import datetime
import importlib
import numpy as np

np.set_printoptions(suppress=True)

In [2]:
# Relative imports
os.chdir("..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir("hidden_state_model")

### Read (and compact) dataframes

In [3]:
compact = True

In [4]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir("data"):
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(f"data/{file}")
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(f"data/{file}", index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 10:
    print("Compacintg dfs")
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = "data/trash"
    for f in read:
        os.rename(f"data/{f}", f"{trash}/{f}")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")

dfs = []  # Clear memory
raw_df

Reading fixed.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,player_name,player_type,action,amount,p,relative_ev,rank,tiebreakers
state_id,,,,,,,,,,,,,,,,,,
4896996752,<NA>,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.5130,0.015390,0,"[8, 4, 0, 0, 0]"
6158719728,4896996752,"[41, 30, 42]","[96, 96]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.6077,0.024308,1,"[4, 8, 3, 2, 0]"
6158687296,6158719728,"[41, 30, 42, 6]","[92, 92]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,8,0.5770,0.046160,1,"[4, 8, 6, 3, 0]"
6158493008,6158687296,"[41, 30, 42, 6, 28]","[84, 84]",0,"[0, 0]","[16, 16]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.5421,0.086736,2,"[4, 2, 8, 0, 0]"
6158678432,6158493008,"[41, 30, 42, 6, 28]","[84, 78]",0,"[0, 6]","[16, 22]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.5421,0.102999,2,"[4, 2, 8, 0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,6127529072,"[8, 9, 36]","[152, 40]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.6240,0.024960,1,"[8, 10, 9, 4, 0]"
6431104400,6431248928,"[8, 9, 36, 6]","[148, 36]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.5459,0.043672,1,"[8, 10, 9, 6, 0]"
6127537840,6431104400,"[8, 9, 36, 6]","[144, 26]",0,"[4, 10]","[12, 18]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.5459,0.081885,1,"[8, 10, 9, 6, 0]"


In [5]:
raw_df.dtypes

prev_entry            object
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
dtype: object

In [6]:
# Check for conflicting rows
dupe_df = raw_df[raw_df.index.duplicated()]
assert len(dupe_df) == 0, dupe_df

## Process data

In [7]:
processor = Processor(raw_df)
df = processor.get_processed_df()
df

,game_id,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,...,opponent_check_river,opponent_check_showdown,action,amount,excess_rank,p,relative_ev,stage,player_name,n_players
4896996752,b95b9e10-00aa-4b52-82e6-a4b220afcffb,0,0,0,0,0,0,0,0,0,...,0,0,call,2,0,0.5130,0.015390,preflop,Tord,2
6158719728,b95b9e10-00aa-4b52-82e6-a4b220afcffb,0,0,0,0,0,2,0,0,0,...,0,0,raise,4,1,0.6077,0.024308,flop,Tord,2
6158687296,b95b9e10-00aa-4b52-82e6-a4b220afcffb,0,4,0,0,0,2,0,0,0,...,0,0,raise,8,1,0.5770,0.046160,turn,Tord,2
6158493008,b95b9e10-00aa-4b52-82e6-a4b220afcffb,0,4,8,0,0,2,0,0,0,...,0,0,check,0,1,0.5421,0.086736,river,Tord,2
6158678432,b95b9e10-00aa-4b52-82e6-a4b220afcffb,0,4,8,0,0,2,0,0,0,...,0,0,call,6,1,0.5421,0.102999,river,Tord,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,6ea05fd6-895d-4791-a676-28ccfc01aca3,0,0,0,0,0,2,0,0,0,...,0,0,raise,4,1,0.6240,0.024960,flop,Tord,2
6431104400,6ea05fd6-895d-4791-a676-28ccfc01aca3,0,4,0,0,0,2,0,0,0,...,0,0,raise,4,1,0.5459,0.043672,turn,Tord,2
6127537840,6ea05fd6-895d-4791-a676-28ccfc01aca3,0,4,4,0,0,2,0,0,0,...,0,0,call,6,1,0.5459,0.081885,turn,Tord,2
6431250896,6ea05fd6-895d-4791-a676-28ccfc01aca3,0,4,4,0,0,2,0,6,0,...,1,0,check,0,0,0.9412,0.169416,river,Tord,2


In [8]:
df.dtypes

game_id                     object
raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn 

## Training

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [10]:
X = df.drop(["excess_rank", "game_id", "p", "relative_ev"], axis=1)
y = df["excess_rank"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [11]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_check_preflop,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,stage,player_name,n_players
4896996752,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,call,2,preflop,Tord,2
6158719728,0,0,0,0,0,2,0,0,0,0,...,0,1,0,0,0,raise,4,flop,Tord,2
6158687296,0,4,0,0,0,2,0,0,0,0,...,0,1,0,0,0,raise,8,turn,Tord,2
6158493008,0,4,8,0,0,2,0,0,0,0,...,0,1,0,0,0,check,0,river,Tord,2
6158678432,0,4,8,0,0,2,0,0,0,0,...,0,1,0,0,0,call,6,river,Tord,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,0,0,0,0,0,2,0,0,0,0,...,0,1,0,0,0,raise,4,flop,Tord,2
6431104400,0,4,0,0,0,2,0,0,0,0,...,0,1,0,0,0,raise,4,turn,Tord,2
6127537840,0,4,4,0,0,2,0,0,0,0,...,0,1,0,0,0,call,6,turn,Tord,2
6431250896,0,4,4,0,0,2,0,6,0,0,...,0,1,0,1,0,check,0,river,Tord,2


In [12]:
y

4896996752    0
6158719728    1
6158687296    1
6158493008    1
6158678432    1
             ..
6431248928    1
6431104400    1
6127537840    1
6431250896    0
6431104448    0
Name: excess_rank, Length: 3741, dtype: int64

In [13]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["action", "stage", "player_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=10_000
            ),
        ),
    ]
)

In [14]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (2988, 35)
Test shape: (753, 35)


In [15]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.7184594953519257


In [16]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_, index=X_test.index)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.7184594953519257
Mean goodness:  0.6587023755736727


,0,1,2,3,4,5,true,pred,correct,goodness
6152138816,0.575456,0.393317,2.437633e-02,0.004199,0.000920,0.001732,0,0,True,0.575456
6152372176,0.492462,0.501762,8.797658e-08,0.005171,0.000009,0.000596,1,1,True,0.501762
6064222992,0.440355,0.553097,1.275816e-08,0.006098,0.000007,0.000442,1,1,True,0.553097
6152371360,0.412266,0.574030,9.235049e-08,0.011754,0.000049,0.001901,1,1,True,0.574030
6152139824,0.343852,0.648316,6.047095e-08,0.006995,0.000025,0.000812,1,1,True,0.648316
...,...,...,...,...,...,...,...,...,...,...
6102526320,0.727166,0.245446,1.855344e-03,0.010136,0.003834,0.011563,0,0,True,0.727166
6127312464,0.653751,0.273855,2.207179e-03,0.022418,0.017274,0.030495,5,0,False,0.030495
6127438960,0.630631,0.234448,9.426888e-04,0.044104,0.038310,0.051565,5,0,False,0.051565
6431193840,0.699688,0.289185,3.372099e-03,0.003866,0.001206,0.002684,0,0,True,0.699688


In [17]:
# Look at incorrect predictions
print(prob_df[prob_df["correct"] == False].shape[0], "incorrect predictions:")
prob_df[prob_df["correct"] == False]

212 incorrect predictions:


,0,1,2,3,4,5,true,pred,correct,goodness
6064258592,0.914019,0.084745,3.873361e-04,0.000416,1.662440e-04,0.000267,1,0,False,0.084745
49cf603d-3b9b-42d7-a2fc-57cf53b5fcb6,0.913266,0.085737,3.051331e-04,0.000187,3.515663e-04,0.000153,1,0,False,0.085737
6064328176,0.624557,0.343195,2.043091e-02,0.002433,6.344308e-03,0.003040,1,0,False,0.343195
6064256288,0.044854,0.567159,3.350623e-01,0.006946,3.578222e-02,0.010197,3,1,False,0.006946
6064224576,0.031556,0.447685,4.705072e-01,0.004643,3.043539e-02,0.015174,3,2,False,0.004643
...,...,...,...,...,...,...,...,...,...,...
10899690400,0.490454,0.506054,1.853470e-17,0.003439,8.294218e-10,0.000053,0,1,False,0.490454
6136911712,0.435146,0.557210,2.175988e-17,0.007506,3.688187e-09,0.000139,0,1,False,0.435146
6095532336,0.460367,0.523181,1.019281e-17,0.016195,8.971002e-09,0.000257,0,1,False,0.460367
6127312464,0.653751,0.273855,2.207179e-03,0.022418,1.727396e-02,0.030495,5,0,False,0.030495


### Attempt to infer card probabilities from rank and table

In [18]:
from cpp_poker.cpp_poker import Hand, CardCollection, HandRank, CheatSheet

In [19]:
hand_ranks_per_state = []
table_ranks = []
for i, (state_id, row) in enumerate(prob_df.iterrows()):
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    table_rank = table_cards.rank_hand().get_rank()
    excess_hand_ranks_row = table_cards.rank_all_hands()
    hand_ranks_per_state.append(
        [rank.get_rank() - table_rank for rank in excess_hand_ranks_row]
    )
    table_ranks.append(table_rank)

table_ranks_df = pd.DataFrame(table_ranks, index=prob_df.index, columns=["table_rank"])
hand_ranks_df = pd.DataFrame(hand_ranks_per_state, index=prob_df.index)
hand_ranks_df

,0,1,2,3,4,5,6,7,8,9,...,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325
6152138816,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6152372176,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
6064222992,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
6152371360,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
6152139824,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6102526320,1,1,1,1,1,1,1,1,2,2,...,0,0,5,5,0,0,0,0,0,8
6127312464,1,1,1,2,1,1,1,1,2,2,...,5,5,5,5,0,5,5,5,5,8
6127438960,1,1,1,2,1,2,1,1,2,2,...,5,5,5,5,0,5,5,5,5,8
6431193840,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
# Get the values of the column indices from hand_ranks_df
column_indices = hand_ranks_df.values

# Use np.arange to create an array of row indices
row_indices = np.arange(hand_ranks_df.shape[0])[:, None]

# Use the row and column indices to index into the DataFrame values
hand_probs_df = pd.DataFrame(
    prob_df.values[row_indices, column_indices],
    index=hand_ranks_df.index,
    columns=hand_ranks_df.columns,
)

# Normalize the hand probabilities to sum to 1 for each row
hand_probs_df = hand_probs_df.div(hand_probs_df.sum(axis=1), axis=0)
hand_probs_df

,0,1,2,3,4,5,6,7,8,9,...,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325
6152138816,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,...,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768
6152372176,0.000785,0.000785,0.00077,0.00077,0.00077,0.00077,0.00077,0.00077,0.000785,0.00077,...,0.00077,0.00077,0.00077,0.00077,0.00077,0.00077,0.00077,0.00077,0.00077,0.00077
6064222992,0.000894,0.000894,0.000712,0.000712,0.000712,0.000712,0.000712,0.000712,0.000894,0.000712,...,0.000712,0.000712,0.000712,0.000712,0.000712,0.000712,0.000712,0.000712,0.000712,0.000712
6152371360,0.000958,0.000958,0.000688,0.000688,0.000688,0.000688,0.000688,0.000688,0.000958,0.000688,...,0.000688,0.000688,0.000688,0.000688,0.000688,0.000688,0.000688,0.000688,0.000688,0.000688
6152139824,0.001098,0.001098,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.001098,0.000582,...,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6102526320,0.000361,0.000361,0.000361,0.000361,0.000361,0.000361,0.000361,0.000361,0.000003,0.000003,...,0.001068,0.001068,0.000017,0.000017,0.001068,0.001068,0.001068,0.001068,0.001068,0.001469
6127312464,0.000679,0.000679,0.000679,0.000005,0.000679,0.000679,0.000679,0.000679,0.000005,0.000005,...,0.000076,0.000076,0.000076,0.000076,0.001622,0.000076,0.000076,0.000076,0.000076,0.0
6127438960,0.000663,0.000663,0.000663,0.000003,0.000663,0.000003,0.000663,0.000663,0.000003,0.000003,...,0.000146,0.000146,0.000146,0.000146,0.001784,0.000146,0.000146,0.000146,0.000146,0.0
6431193840,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,...,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781


In [21]:
# Validate that rows sum to 1
hand_probs_df.sum(axis=1)

6152138816    1.0
6152372176    1.0
6064222992    1.0
6152371360    1.0
6152139824    1.0
             ... 
6102526320    1.0
6127312464    1.0
6127438960    1.0
6431193840    1.0
6127534384    1.0
Length: 753, dtype: object

In [22]:
# Look at the max probabilites for different hands
mask = prob_df["pred"] == 2
max_probs = hand_probs_df[mask].max(axis=1)
min_probs = hand_probs_df[mask].min(axis=1)
max_prob_hands = hand_probs_df[mask].idxmax(axis=1)
min_prob_hands = hand_probs_df[mask].idxmin(axis=1)
mean_probs = hand_probs_df[mask].mean(axis=1)
sample_prob_df = pd.DataFrame(
    {
        "max_prob": max_probs,
        "max_prob_hand": max_prob_hands,
        "min_prob": min_probs,
        "min_prob_hand": min_prob_hands,
        "mean_prob": mean_probs,
    }
)
sample_prob_df["max_prob_hand"] = sample_prob_df["max_prob_hand"].apply(lambda x: Hand(x).str())
sample_prob_df["min_prob_hand"] = sample_prob_df["min_prob_hand"].apply(lambda x: Hand(x).str())
sample_prob_df["table"] = raw_df.loc[sample_prob_df.index, "public_cards"].apply(
    lambda x: CardCollection(x).str()
)
sample_prob_df["predicted_excess_rank"] = prob_df.loc[sample_prob_df.index, "pred"]
sample_prob_df["pred_rank"] = (
    sample_prob_df["predicted_excess_rank"]
    + table_ranks_df.loc[sample_prob_df.index, "table_rank"]
)
sample_prob_df["pred_rank_str"] = sample_prob_df["pred_rank"].apply(
    lambda x: HandRank(x, []).get_rank_name()
)
sample_prob_df = sample_prob_df.drop(["predicted_excess_rank"], axis=1)
sample_prob_df

,max_prob,max_prob_hand,min_prob,min_prob_hand,mean_prob,table,pred_rank,pred_rank_str
6064224576,0.000875,♥ 2 ♥ 3,0.000129,♥ 2 ♥ A,0.000754,♥ 7 ♦ 7 ♣ 7 ♠ A,5,Flush
6064257536,0.005945,♥ 2 ♠ 7,0.000499,♥ 2 ♥ 3,0.000754,♥ 7 ♦ 7 ♣ 7 ♠ A,5,Flush
5803868512,0.002783,♥ 5 ♥ 6,0.000002,♥ 2 ♥ 3,0.000754,♥ 4 ♦ J ♣ 5 ♣ 6,2,Two Pairs
6165563904,0.003191,♥ 9 ♥ J,0.000042,♦ 2 ♦ 3,0.000754,♦ 9 ♦ J ♦ A,2,Two Pairs
6085606656,0.002888,♥ 2 ♥ 5,0.000022,♥ 2 ♦ 2,0.000754,♥ 9 ♦ 5 ♣ 2 ♣ K,2,Two Pairs
6087820048,0.004509,♥ 2 ♥ 5,0.000012,♥ 2 ♦ 2,0.000754,♥ 9 ♦ 5 ♣ 2 ♣ K,2,Two Pairs
6125188576,0.001414,♥ 7 ♥ 8,0.000004,♥ 7 ♦ 7,0.000754,♥ 3 ♥ 9 ♦ 8 ♠ 7,2,Two Pairs


In [23]:
# Deepdive into a specific row
state_id = "6165563904"
row = sample_prob_df.loc[state_id]
most_likely_hands = hand_probs_df.loc[state_id].sort_values(ascending=False).head(5)
least_likely_hands = hand_probs_df.loc[state_id].sort_values(ascending=True).head(5)
print("Table cards:", row["table"])
print("Predicted excess rank:", prob_df.loc[state_id, "pred"])
print("Actual excess rank", df.loc[state_id, "excess_rank"])
print("Most and least likely hands:")
for hand, prob in most_likely_hands.items():
    print(f"{Hand(hand).str()}: {prob*100:.2f}%")
print("...")
for hand, prob in least_likely_hands.items():
    print(f"{Hand(hand).str()}: {prob*100:.2f}%")

Table cards: ♦ 9 ♦ J ♦ A 
Predicted excess rank: 2
Actual excess rank 1
Most and least likely hands:
♣ 9 ♠ J : 0.32%
♥ A ♠ J : 0.32%
♣ A ♠ J : 0.32%
♣ 9 ♠ A : 0.32%
♥ A ♣ 9 : 0.32%
...
♦ 4 ♦ 7 : 0.00%
♦ 7 ♦ K : 0.00%
♦ 10 ♦ K : 0.00%
♦ 10 ♦ Q : 0.00%
♦ 4 ♦ Q : 0.00%


### Attempt to infer winning probabilities from hidden state probabilities

In [24]:
hand_winning_probs = []
for i, (state_id, row) in enumerate(hand_probs_df.iterrows()):
    print(f"Processing row {i}", flush=True)
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    processed_row = df.loc[state_id]
    n_players = processed_row["n_players"]
    hand_winning_probs.append(CheatSheet.get_all_winning_probabilities(table_cards, n_players, 1000))

hand_winning_probs_df = pd.DataFrame(hand_winning_probs, index=hand_probs_df.index)
hand_winning_probs_df

Processing row 0
